In [0]:
%%capture --no-stderr
%pip install --quiet -U langchain_core langgraph databricks-langchain
dbutils.library.restartPython()

In [0]:
import os
os.environ['LANGSMITH_API_KEY'] = dbutils.secrets.get(scope="eo_scope", key="LANGSMITH_KEY")
os.environ['LANGSMITH_TRACING'] = "true"
os.environ['LANGSMITH_TRACING_V2'] = "true"
os.environ['LANGSMITH_PROJECT'] = "langchain-academy-module2"
os.environ['LANGSMITH_ENDPOINT'] = "https://api.smith.langchain.com"

In [0]:
from databricks_langchain import ChatDatabricks
from langgraph.graph import StateGraph, START, END, MessagesState
from langchain_core.messages import SystemMessage, HumanMessage, RemoveMessage

class State(MessagesState):
    summary: str

llm = ChatDatabricks(model='databricks-meta-llama-3-3-70b-instruct')

In [0]:
def assistant_node(state: State) -> State:

  summary = state.get('summary', '')

  if summary:

    system_message = f'Summary of the conversation so far: {summary}'
    system_prompt = SystemMessage(content=system_message)
    return {'messages': llm.invoke([system_prompt] + state['messages'])}
  
  return {'messages': llm.invoke(state['messages'])}

def summarization_node(state: State) -> State:
  
  summary = state.get('summary', '')

  if summary:

    summary_message = f'This is the summary of the conversation so far: {summary}\n\n'
    'Extend the summary by taking into account the new messages above.'

  else:
    summary_message = 'Summerize the conversation above, but maintain all the key information.'

  messages = state["messages"] + [HumanMessage(content=summary_message)]
  response = llm.invoke(messages)
  
  delete_messages = [RemoveMessage(id=m.id) for m in state['messages'][:-2]]
  return {"summary": response.content, 'messages': delete_messages}

In [0]:
from typing import Literal
def should_summarize(state: State) -> Literal['Summarization', END]:
  
  messages = state["messages"]
  
  if len(messages) > 6:
    return 'Summarization'
  
  return END

In [0]:
from langgraph.checkpoint.memory import MemorySaver
memory = MemorySaver()

graph = StateGraph(State)

graph.add_node('Assistant', assistant_node)
graph.add_node('Summarization', summarization_node)

graph.add_edge(START, 'Assistant')
graph.add_conditional_edges('Assistant', should_summarize)
graph.add_edge('Assistant', END)

app = graph.compile(checkpointer=memory)
app

In [0]:
config = {'configurable': {"thread_id": "1"}}

# Start conversation
input_message = HumanMessage(content="hi! I'm Lance")
output = app.invoke({"messages": [input_message]}, config) 
for m in output['messages'][-1:]:
    m.pretty_print()

input_message = HumanMessage(content="what's my name?")
output = app.invoke({"messages": [input_message]}, config) 
for m in output['messages'][-1:]:
    m.pretty_print()

input_message = HumanMessage(content="i like the 49ers!")
output = app.invoke({"messages": [input_message]}, config) 
for m in output['messages'][-1:]:
    m.pretty_print()

In [0]:
app.get_state(config).values.get('summary', '')

In [0]:
input_message = HumanMessage(content="i like Nick Bosa, isn't he the highest paid defensive player?")
output = app.invoke({"messages": [input_message]}, config) 
for m in output['messages'][-1:]:
    m.pretty_print()

In [0]:
app.get_state(config).values.get("summary","")